In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.00),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  2750.00),
    ("Eve",   "Savings",   150.00),
]

columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

df.show()            # print the table
# df.printSchema()   # show column names + types
# df.count()         # count rows (returns a number)

# --- Summing a column ---
df.agg(F.sum("balance")).show()                    # total of all balances

# --- Sum with a WHERE clause ---
(df.filter(F.col("balance") > 1000.00)
   .agg(F.sum("balance"))
   .show())                                        # total of balances over 1000

# --- Sum with two conditions (AND) ---
(df.filter((F.col("balance") > 1000.00) & (F.col("account_type") == "Savings"))
   .agg(F.sum("balance"))
   .show())                                        # Savings balances over 1000

# --- Filter rows where the name contains a lowercase "e" ---
(df.filter((F.col("balance") > 1000.00) & (F.col("name").contains("e")))
   .agg(F.sum("balance"))
   .show())                                        # over-1000 with an "e" in the name

# --- Just show matching rows instead of summing ---
(df.filter(F.col("name").contains("e"))
   .show())                                        # people with an "e" in their name


In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.96),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  2750.00),
    ("Eve",   "Savings",   -150.00),
]


columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

df.select("name", "balance").show() 


(df.withColumn("with_interest", F.col("balance") * 1.05)
   .show())


(df.withColumn("tier",
        F.when(F.col("balance") >= 5000, "Gold")
         .when(F.col("balance") >= 1000, "Silver")
         .otherwise("Bronze"))
   .show())


df.select("name", "balance").show() 

df.withColumn("balance_rounded", F.round( F.col("balance"),0)).show()

df.withColumn("active", F.when (F.col("balance") > 0, "Yes")
              .otherwise("No")).show()


In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.96),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  12750.00),
    ("Eve",   "Savings",   -150.00),
]


columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

(df.groupBy("account_type")
   .count()
   .show())


(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .show())


(df.groupBy("account_type")
   .agg(
       F.count("*").alias("num_accounts"),
       F.sum("balance").alias("total_balance"),
       F.round(F.avg("balance"),2).alias("avg_balance"),
       F.min("balance").alias("min_balance"),
       F.max("balance").alias("max_balance"),
   )
   .show())


(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .filter(F.col("total_balance") > 2000)
   .show())


(df.groupBy("account_type")
   .count()
   .show())



(df.groupBy("account_type")
   .agg(
       F.round (F.avg("balance"),2).alias("Avg_balance"),
       F.round(F.sum("balance"),2).alias("total_balance"),
       F.max("balance").alias("max_balance")
   )
   .show())

(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .filter(F.col("total_balance") > 10000)
   .show())

In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with categories (like our 'tier' column earlier)
data = [
    ("Alice", "Gold", 6000),
    ("Bob", "Silver", 2500),
    ("Charlie", "Bronze", 500),
    ("Diana", "Gold", 8000),
    ("Evan", "Silver", 1500),
    ("Fiona", "Bronze", 800)
]

columns = ["name", "tier", "balance"]
df = spark.createDataFrame(data, columns)

# 2. Group by tier and calculate metrics (Total balance, Average balance, and Customer count)
summary_df = df.groupBy("tier").agg(
    F.sum("balance").alias("total_balance"),
    F.round(F.avg("balance"), 2).alias("avg_balance"),
    F.count("name").alias("customer_count")
)

# 3. Show the resulting aggregated DataFrame
summary_df.show()


from pyspark.sql import functions as F

# 1. Group by tier, filter for groups with more than 1 customer, and sort by total balance descending
filtered_summary_df = (
    df.groupBy("tier")
    .agg(
        F.sum("balance").alias("total_balance"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.count("name").alias("customer_count")
    )
    .filter(F.col("avg_balance") >= 1000)
    .orderBy(F.col("total_balance").asc())
)

filtered_summary_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create a customer accounts DataFrame
accounts_data = [
    (101, "Alice", "Gold"),
    (102, "Bob", "Silver"),
    (103, "Charlie", "Bronze"),
    (104, "Diana", "Gold")
]
accounts_df = spark.createDataFrame(accounts_data, ["account_id", "name", "tier"])

# 2. Create a separate transactions DataFrame
transactions_data = [
    (101, 500),
    (101, 1200),
    (102, 300),
    (105, 999) # ID 105 doesn't exist in accounts
]
transactions_df = spark.createDataFrame(transactions_data, ["account_id", "transaction_amount"])

# 3. Perform a Left Join to keep all accounts and match their transactions
joined_df = accounts_df.join(
    transactions_df, 
    on="account_id", 
    how="left"
)

joined_df.show()